# Dates, times, and durations · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 13. Dates, times, and durations

Parse dates explicitly and use calendar-aware operations.

Before you start: You can convert types and select rows by conditions.

Official reference: https://pandas.pydata.org/docs/user_guide/timeseries.html

### M13C01. Turn date text into dates

A date-looking string is still text until it is parsed. `to_datetime()` converts it to a date/time type. An explicit format removes ambiguity; impossible dates can be marked as NaT, pandas missing date value.

**Read:** What do %Y, %m, and %d represent?

Answer: Four-digit year, numeric month, and day of month.

**Predict:** What happens to February 30?

Answer: It becomes NaT because it is not a valid calendar date.

**Explain:** Why might 01/02/2026 be ambiguous without a format?

Answer: It could mean January 2 or February 1, depending on the intended date convention.



In [ ]:
# Predict before running: What happens to February 30?
source = pd.Series(["2026-09-01", "2026-09-03", "2026-02-30"], dtype="string")
result = pd.to_datetime(source, format="%Y-%m-%d", errors="coerce")
print(result)
print("Unparsed dates:", result.isna().sum())

**Your turn:** Parse ["01/09/2026", "03/09/2026", "30/02/2026"] as day/month/year. Count invalid dates.

For the check, keep these result names: `result`.

**Hint:** Use the explicit format "%d/%m/%Y".

**Watch for this:** Inspect parsing failures; errors="coerce" does not repair the source date.

In [ ]:
# Change this code to complete the task.
# Question: Parse ["01/09/2026", "03/09/2026", "30/02/2026"] as day/month/year. Count invalid dates.
source = pd.Series(["01/09/2026", "03/09/2026", "30/02/2026"], dtype="string")
result = pd.to_datetime(source, format="%d/%m/%Y", errors="coerce")
print(result)
print("Unparsed dates:", result.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.iloc[0] == pd.Timestamp("2026-09-01")
assert result.iloc[1].day == 3
assert result.isna().sum() == 1
print("Code checks passed. Review the written explanation as well.")

The first two dates are September 1 and September 3. February 30 remains invalid.

**Quick check:** What is NaT?

1. A missing datetime value
2. A date-format instruction
3. The current time

Answer: 1. NaT marks a missing or unparseable date/time value.

**Explain what you learned:** Parse known date formats explicitly.

### M13C02. Extract useful calendar parts

After parsing dates, the `.dt` accessor exposes calendar parts such as year, month, and weekday. These features are useful for grouping and checking patterns. Do not try to use `.dt` on unparsed strings.

**Read:** Why can .dt be used on Join_Date here?

Answer: The column has been converted from text into a datetime type.

**Predict:** Will all Month values be the same in this dataset?

Answer: Yes, all join dates are in September, so every Month is 9.

**Explain:** Why might a month number need a year too in a larger dataset?

Answer: September from different years would otherwise be grouped together.



In [ ]:
# Predict before running: Will all Month values be the same in this dataset?
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
result = df[["Student_ID", "Join_Date"]].copy()
result["Month"] = result["Join_Date"].dt.month
result["Weekday"] = result["Join_Date"].dt.day_name()
print(result.head(4))

**Your turn:** Add Year and Day columns for the same join dates. Store only the first three rows in result, with columns Student_ID, Year, Month, and Day, then print result.

For the check, keep these result names: `result`.

**Hint:** Use `.dt.year`, `.dt.month`, and `.dt.day`.

**Watch for this:** A weekday pattern in twelve fictional records is a practice result, not evidence about real student behavior.

In [ ]:
# Change this code to complete the task.
# Question: Add Year and Day columns for the same join dates. Store only the first three rows in result, with columns Student_ID, Year, Month, and Day, then print result.
df = students()
dates = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
result = pd.DataFrame({"Student_ID": df["Student_ID"], "Year": dates.dt.year, "Month": dates.dt.month, "Day": dates.dt.day}).head(3)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Day"].tolist() == [1, 2, 3]
assert result["Year"].eq(2026).all()
print("Code checks passed. Review the written explanation as well.")

The years are 2026, the months are 9, and the day values are 1, 2, and 3.

**Quick check:** Which accessor provides calendar components?

1. .str
2. .dt
3. .iloc

Answer: 2. .dt exposes datetime properties and methods.

**Explain what you learned:** Create date features after parsing the date type.

### M13C03. Filter an explicit date interval

Date comparisons let you select a time interval. `between()` includes both boundaries by default; naming the inclusive rule makes your intent clearer. Here every date is a midnight date with no separate time-of-day information.

**Read:** Are September 3 and September 6 included?

Answer: Yes; inclusive="both" includes both endpoints.

**Predict:** Which IDs appear?

Answer: S003, S004, S005, and S006.

**Explain:** Why do interval boundary rules matter?

Answer: An included or excluded endpoint changes which records contribute to the analysis.



In [ ]:
# Predict before running: Which IDs appear?
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
mask = df["Join_Date"].between("2026-09-03", "2026-09-06", inclusive="both")
result = df.loc[mask, ["Student_ID", "Join_Date"]]
print(result)

**Your turn:** Select dates on or after September 3 and strictly before September 6 using two comparisons.

For the check, keep these result names: `result`.

**Hint:** Combine >= and < with & and parentheses.

**Watch for this:** With timestamps containing hours, an end date at midnight is not the whole final day; a next-day exclusive bound is often clearer.

In [ ]:
# Change this code to complete the task.
# Question: Select dates on or after September 3 and strictly before September 6 using two comparisons.
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
mask = (df["Join_Date"] >= "2026-09-03") & (df["Join_Date"] < "2026-09-06")
result = df.loc[mask, ["Student_ID", "Join_Date"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Student_ID"].tolist() == ["S003", "S004", "S005"]
print("Code checks passed. Review the written explanation as well.")

S003, S004, and S005 remain. The exclusive upper boundary removes S006.

**Quick check:** Which operator excludes the upper boundary?

1. <=
2. <
3. >=

Answer: 2. A strict less-than comparison excludes an equal endpoint.

**Explain what you learned:** State the start and end rules of every date filter.

### M13C04. Calculate elapsed time with a fixed reference

Subtracting dates produces timedeltas, which represent durations. `.dt.days` extracts whole days from those durations. A fixed reference date makes a classroom result reproducible when the file is opened later.

**Read:** What object is produced by reference - joined?

Answer: A Series of time durations, called timedeltas.

**Predict:** How many elapsed days does S001 have on October 1?

Answer: 30 days from September 1 to October 1.

**Explain:** Why use a fixed reference instead of today()?

Answer: Everyone can reproduce the same answer, regardless of when they run the lesson.



In [ ]:
# Predict before running: How many elapsed days does S001 have on October 1?
df = students()
joined = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
reference = pd.Timestamp("2026-10-01")
result = pd.DataFrame({"Student_ID": df["Student_ID"], "Days_enrolled": (reference - joined).dt.days})
print(result.head(4))

**Your turn:** Change the reference to October 8 and select students with at least 30 elapsed days enrolled.

For the check, keep these result names: `result`.

**Hint:** Recalculate the duration, then filter Days_enrolled >= 30.

**Watch for this:** For real timestamps, keep time zones consistent before subtracting them.

In [ ]:
# Change this code to complete the task.
# Question: Change the reference to October 8 and select students with at least 30 elapsed days enrolled.
df = students()
joined = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
reference = pd.Timestamp("2026-10-08")
df["Days_enrolled"] = (reference - joined).dt.days
result = df.loc[df["Days_enrolled"] >= 30, ["Student_ID", "Days_enrolled"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 8
assert result.iloc[-1]["Student_ID"] == "S008"
assert result.iloc[-1]["Days_enrolled"] == 30
print("Code checks passed. Review the written explanation as well.")

S001 through S008 meet the rule. S008 has exactly 30 elapsed days.

**Quick check:** Does elapsed time count both the start and end date as full extra days?

1. Yes
2. No; it is the duration between the two dates
3. Only during September

Answer: 2. Date subtraction measures elapsed time. Inclusive calendar-day counts are a separate reporting rule.

**Explain what you learned:** Use timedelta arithmetic instead of subtracting date strings.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates